# 04 · Handwriting OCR — the vision path (currently broken)

**Read this before running anything below.** This notebook's transcription
call is expected to fail. It calls Groq's vision API with the model named
in the `GROQ_VISION_MODEL` environment variable, defaulting to
**`meta-llama/llama-4-scout-17b-16e-instruct`**. Groq retired that model on
**2026-08-16** with no replacement configured here, and this path has not
worked since — stated plainly instead of discovered on day three.

**In → out (when it works):** a photo or scan of a handwritten page → plain
transcribed text, with `[FIGURE: ...]` placeholders for diagrams and a
second pass that turns each placeholder into an SVG.

**This is a good first issue.** The fix is bounded and checkable: pick a
vision-capable model that is currently live (on Groq, or point the same
call shape at a different provider's vision endpoint), set
`GROQ_VISION_MODEL` (and `GROQ_API_KEY`) accordingly, and confirm it
transcribes `sample-data/handwriting-sample.png` correctly. You do not need
to understand any other part of this repository to do that.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `_groq_api_key` / `_groq_headers` | Reads `GROQ_API_KEY` from the environment and builds the request headers for Groq's chat API | `_groq_headers()` |
| `_image_to_jpeg_b64` | Re-encodes an image file's bytes as a base64 JPEG for the vision API payload | `_image_to_jpeg_b64(image_bytes)` |
| `_call_groq_vision` | POSTs one image + prompt to Groq's vision chat endpoint and returns the model's text | `_call_groq_vision(b64_jpeg, TRANSCRIBE_PROMPT)` |
| `_call_groq_text` | POSTs a text-only prompt to Groq's chat endpoint (used for the SVG pass) | `_call_groq_text(SVG_PROMPT.format(description=...))` |
| `_pass1_transcribe` | Wraps `_call_groq_vision` with the transcription prompt and a language hint | `_pass1_transcribe(image_bytes, language="auto")` |
| `_pass2_svg` | Wraps `_call_groq_text` with the SVG prompt for one detected `[FIGURE: ...]` | `_pass2_svg("a right triangle with legs 3 and 4")` |
| `_extract_page` | Runs pass 1, finds every `[FIGURE: ...]` placeholder, runs pass 2 on each, returns one page record | `_extract_page(image_bytes, page_no=1, filename=..., language="auto")` |
| `merge_pages_to_text` | Joins multiple page records into one `[Page N]`-labelled text blob | `merge_pages_to_text([page])` |
| `_compile_pitfalls` | Compiles a small YAML rule set into conditioned regex substitutions | `_compile_pitfalls(_EXAMPLE_PITFALLS_YAML)` |
| `apply_ocr_pitfalls` | Applies compiled pitfall rules to transcribed text, line by line | `apply_ocr_pitfalls(text, rules)` |

## Step 1 — locate the repo root and confirm the environment

Before anything else, resolve `REPO_ROOT` (the kernel's cwd is this notebook's own directory, not the repo root) and print which API keys — including `GROQ_VISION_MODEL` — are present, so the offline/no-key path this notebook takes is stated up front rather than discovered by a later failure.

In [ ]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from __future__ import annotations

import os
from pathlib import Path

import nbio

REPO_ROOT = nbio.bootstrap()
nbio.show_environment(extra_keys=["GROQ_VISION_MODEL"])

## Step 2 — rasterize a PDF page to pixels, if the input is one

`SUPPORTED_EXTENSIONS` below claims `.pdf`, but Pillow's `Image.open` cannot read one -- there was no rasterizer in this notebook to back that claim, so a `.pdf` input would crash. `03-orientation.ipynb`, in this same stage folder, already has the fix: three lines with PyMuPDF (`fitz`) that turn a PDF page into pixels. Copied verbatim from there, not from any of the three other copies of the same three lines elsewhere in the lab -- they're the same pattern again, and copying any of them here would just add a fifth.

In [ ]:
import fitz


def rasterize_pdf_first_page(pdf_path: Path, zoom: float = 2.0) -> bytes:
    """PDF page one -> PNG bytes. Same three-line pattern as
    03-orientation.ipynb's `fitz.open(...).get_pixmap(...)` -- copied,
    not reimplemented, since a fourth copy of this in the lab (after
    terrier-ta's claude_ocr_service.py, its material_lookup.py, and
    docling's links.py) would be the fifth.
    """
    doc = fitz.open(str(pdf_path))
    pix = doc[0].get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    png_bytes = pix.tobytes("png")
    doc.close()
    return png_bytes


print("rasterize_pdf_first_page defined -- not exercised by this "
      "notebook's own sample, which is already a PNG. Point SAMPLE_IMAGE "
      "at a .pdf (Step 3, next) to exercise this path.")

## Step 3 — the sample image

`sample-data/handwriting-sample.png` is **not** real handwriting — it's a
synthetic, machine-rendered page (italic system font, a few pixels of
per-line jitter) built for this repo so nothing here is a real student's
work. See the stage `README.md` for why. It mimics the shape
`claude_ocr_service.py`'s prompt was written for: a worked maths answer with
a `Sol:` marker and a starred final answer.

This sample is already a PNG, so the branch below reads it straight in —
the `.pdf` branch, using `rasterize_pdf_first_page` from Step 2, only runs
if `SAMPLE_IMAGE` is pointed at a `.pdf` instead.

In [ ]:
import io

from PIL import Image

SAMPLE_IMAGE = Path("sample-data/handwriting-sample.png").resolve()

if SAMPLE_IMAGE.suffix.lower() == ".pdf":
    sample_image = Image.open(io.BytesIO(rasterize_pdf_first_page(SAMPLE_IMAGE)))
else:
    sample_image = Image.open(SAMPLE_IMAGE)

sample_image


## Step 4 — the call's configuration: URL, models, prompts

`GROQ_VISION_MODEL`/`GROQ_MODEL` are read live from `os.environ` at call
time, so a model swap is a `.env` edit, not a code hunt — these constants
are only the *defaults* used when those env vars are unset. Both prompts
(transcribe, then render any detected figure as SVG) live here too.

In [ ]:
import base64
import io
import json
import re

import httpx

GROQ_CHAT_URL = "https://api.groq.com/openai/v1/chat/completions"

# NOTE: meta-llama/llama-4-scout-17b-16e-instruct was retired by Groq on
# 2026-08-16. This default is kept as-is so the failure below is the real
# failure, not one masked by quietly picking a different model on its
# behalf.
DEFAULT_GROQ_VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
DEFAULT_GROQ_TEXT_MODEL = "llama-3.1-8b-instant"

SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp", ".webp"}
SUPPORTED_LANGUAGES = {"hindi", "punjabi", "english", "sanskrit", "auto"}

_FIGURE_RE = re.compile(r"\[FIGURE:\s*(?P<desc>[^\]]+)\]")
_SVG_RE = re.compile(r"<svg[\s\S]*?</svg>", re.DOTALL)

TRANSCRIBE_PROMPT = """
This is a scanned page of a student's handwritten maths exam answers.

Transcribe everything on the page exactly as written. Requirements:
1. Preserve structure: Question label, "Sol:" marker, working steps, final answer.
2. Write every mathematical expression clearly using Unicode symbols:
   π  ×  ÷  √  ²  ³  °  ½  ¼  ∴
   Fractions as: numerator / denominator
3. Mark each final boxed or underlined answer with ★.
4. For any diagram or figure on the page, output a placeholder on its own line:
   [FIGURE: <precise description — shapes, labels, dimensions, angles>]
5. Output plain text only — no markdown, no HTML. Use blank lines to separate sections.
""".strip()

SVG_PROMPT = """
Generate an SVG diagram for a student's maths exam answer sheet.

Figure description: {description}

Output ONLY a single <svg> element — no explanation, no markdown fences.
Requirements:
- viewBox="0 0 260 260" width="260" height="260"
- White background: <rect width="260" height="260" fill="#fff" rx="4"/>
- Lines/curves: stroke="#1a3a6a" fill="none" stroke-width="2"
- Filled regions (shaded areas): fill="#d0e4f7" stroke="#1a3a6a" stroke-width="2"
- Right-angle marker: 12x12 square at the corner, stroke="#1a3a6a" fill="none"
- Labels: font-family="Georgia,serif" font-size="14" fill="#1a1a1a"
- Center figure with at least 30px margin on all sides
""".strip()

# populated as a side effect by _call_groq_vision/_call_groq_text below,
# read by the spend-ceiling cell after a real call
_LAST_USAGE: dict = {}


## Step 5 — reading the API key and building request headers

Two one-line helpers, defined together since `_groq_headers` is nothing
but `_groq_api_key` plus the fixed content-type header.

In [ ]:
def _groq_api_key() -> str:
    return os.environ.get("GROQ_API_KEY", "")


def _groq_headers() -> dict:
    return {
        "Authorization": f"Bearer {_groq_api_key()}",
        "Content-Type": "application/json; charset=utf-8",
    }

## Step 6 — re-encoding the image for the API payload

Groq's vision endpoint expects a base64 JPEG data URL, regardless of the
sample image's own format.

In [ ]:
def _image_to_jpeg_b64(image_bytes: bytes) -> str:
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=95)
    return base64.b64encode(buf.getvalue()).decode()

## Step 7 — the vision call

POSTs one image + prompt to Groq's chat-completions endpoint with the
vision model in play. This is the call that is expected to fail once
invoked below — the model it defaults to no longer exists on Groq's side.

In [ ]:
def _call_groq_vision(b64_jpeg: str, prompt: str, timeout: int = 150) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_jpeg}"}},
            {"type": "text", "text": prompt},
        ],
    }]
    payload = json.dumps({
        "model": os.environ.get("GROQ_VISION_MODEL", DEFAULT_GROQ_VISION_MODEL),
        "messages": messages,
        "max_tokens": 8192,
        "temperature": 0.1,
    }, ensure_ascii=False).encode("utf-8")
    with httpx.Client(timeout=httpx.Timeout(float(timeout))) as client:
        r = client.post(GROQ_CHAT_URL, headers=_groq_headers(), content=payload)
    if r.status_code >= 400:
        raise RuntimeError(f"Groq vision API {r.status_code}: {r.text[:600]}")
    body = r.json()
    usage = body.get("usage", {})
    _LAST_USAGE["model"] = os.environ.get("GROQ_VISION_MODEL", DEFAULT_GROQ_VISION_MODEL)
    _LAST_USAGE["prompt_tokens"] = usage.get("prompt_tokens", 0)
    _LAST_USAGE["completion_tokens"] = usage.get("completion_tokens", 0)
    return body["choices"][0]["message"]["content"].strip()

## Step 8 — the text call

Same shape as the vision call, minus the image — used for the second pass
that turns a `[FIGURE: ...]` description into SVG.

In [ ]:
def _call_groq_text(prompt: str, timeout: int = 80) -> str:
    messages = [{"role": "user", "content": prompt}]
    payload = json.dumps({
        "model": os.environ.get("GROQ_MODEL", DEFAULT_GROQ_TEXT_MODEL),
        "messages": messages,
        "max_tokens": 4096,
        "temperature": 0.2,
    }, ensure_ascii=False).encode("utf-8")
    with httpx.Client(timeout=httpx.Timeout(float(timeout))) as client:
        r = client.post(GROQ_CHAT_URL, headers=_groq_headers(), content=payload)
    if r.status_code >= 400:
        raise RuntimeError(f"Groq text API {r.status_code}: {r.text[:600]}")
    body = r.json()
    usage = body.get("usage", {})
    _LAST_USAGE["model"] = os.environ.get("GROQ_MODEL", DEFAULT_GROQ_TEXT_MODEL)
    _LAST_USAGE["prompt_tokens"] = usage.get("prompt_tokens", 0)
    _LAST_USAGE["completion_tokens"] = usage.get("completion_tokens", 0)
    return body["choices"][0]["message"]["content"].strip()

## Step 9 — pass 1: transcribe

Wraps `_call_groq_vision` with the transcription prompt and a language
hint (`auto` asks the model to detect Hindi/Punjabi/English/Sanskrit
itself).

In [ ]:
def _pass1_transcribe(image_bytes: bytes, language: str = "auto") -> str:
    hint = (
        "Primary language hint: auto-detect Hindi/Punjabi/English/Sanskrit."
        if language == "auto" else f"Primary language hint: {language}."
    )
    b64 = _image_to_jpeg_b64(image_bytes)
    return _call_groq_vision(b64, f"{TRANSCRIBE_PROMPT}\n\n{hint}", timeout=150)

## Step 10 — pass 2: render one figure placeholder as SVG

Wraps `_call_groq_text` with the SVG prompt and pulls the `<svg>...</svg>`
element back out of whatever the model returned around it.

In [ ]:
def _pass2_svg(figure_description: str) -> str:
    raw = _call_groq_text(SVG_PROMPT.format(description=figure_description), timeout=80)
    m = _SVG_RE.search(raw)
    return m.group(0).strip() if m else ""

## Step 11 — `_extract_page`, tying pass 1 and pass 2 together

Transcribes the page, finds every `[FIGURE: ...]` placeholder pass 1 left
behind, runs pass 2 on each one, and returns a single page record.

In [ ]:
def _extract_page(image_bytes: bytes, page_no: int, filename: str, language: str) -> dict:
    text = _pass1_transcribe(image_bytes, language).strip()
    figures = []
    for i, m in enumerate(_FIGURE_RE.finditer(text), start=1):
        desc = m.group("desc").strip()
        figures.append({"index": i, "description": desc, "svg": _pass2_svg(desc)})
    return {"page": page_no, "text": text, "language": language, "source_file": filename, "figures": figures}

## Step 12 — `merge_pages_to_text`

A multi-page document would produce one record per page from
`_extract_page`; this joins them into one `[Page N]`-labelled blob.

In [ ]:
def merge_pages_to_text(pages: list[dict]) -> str:
    parts = []
    for page in pages:
        t = str(page.get("text", "")).strip()
        if t:
            parts.append(f"[Page {page.get('page', '?')}]\n{t}")
    return "\n\n".join(parts)

## Step 13 — attempting the call, under a spend ceiling

This is the one notebook in this stage that can call a paid model, so it is the one that gets a ceiling. `nbio.cost_meter` measures real spend against the calls made inside its `with` block and raises `BudgetExceeded` the moment spend crosses the budget — before a retried or looped call turns into a bill nobody was watching. 50 cents is sized to catch a mistake, not to ration a normal run: a single page's worth of vision + text calls costs a small fraction of it. Still wrapped in a `try`/`except`, same as before, so this cell records a clean, stated reason rather than an uncaught stack trace either way.

In [ ]:
image_bytes = SAMPLE_IMAGE.read_bytes()

with nbio.cost_meter(budget_usd=0.50) as meter:
    try:
        if not _groq_api_key():
            raise RuntimeError(
                "GROQ_API_KEY is not set. This is the first thing to fix -- add it "
                "to a .env file at the repo root (never commit it; see .gitignore) "
                "-- but setting it alone will not make this call succeed: see the "
                "model note below."
            )
        page = _extract_page(image_bytes, page_no=1, filename=SAMPLE_IMAGE.name, language="auto")
        if _LAST_USAGE:
            meter.record(_LAST_USAGE["model"], _LAST_USAGE["prompt_tokens"], _LAST_USAGE["completion_tokens"])
        print("Transcription succeeded -- the model in GROQ_VISION_MODEL is live:")
        print(page["text"])
    except Exception as exc:
        print(f"Expected failure ({type(exc).__name__}): {exc}")
        print()
        print("To try a fix: set GROQ_API_KEY and")
        print(f"  GROQ_VISION_MODEL=<a currently-live Groq vision model id>")
        print("in a .env file at the repo root, then re-run this cell.")

print()
print(meter.report())


## Step 14 — compiling a pitfalls YAML rule

Regex fixes for recurring transcription bugs (e.g. a specific
`(22/7) × (1/4) ×` duplication misread) are naturally tuned to the
originating dataset's own quirks, so the full registry doesn't belong in a
public sample repo. The loading *mechanism* — YAML → compiled, conditioned
regex substitutions, applied per line — is kept in full; what's inline
below is a small illustrative rule for this sample's own content instead
of a full registry.

In [ ]:
import yaml

_EXAMPLE_PITFALLS_YAML = """
pitfalls:
  - id: asterisk_as_final_answer_marker
    description: Some vision models emit a bare "*" instead of the requested "★" for a boxed answer.
    enabled: true
    when_line_matches_all:
      - "Final answer"
    pattern: '\\*\\s*$'
    replacement: "★"
    count_per_line: 1
"""


def _compile_pitfalls(yaml_text: str) -> list[dict]:
    raw = yaml.safe_load(yaml_text) or {}
    rules = []
    for p in raw.get("pitfalls") or []:
        if not p.get("enabled", True):
            continue
        when_all = [re.compile(w) for w in p.get("when_line_matches_all") or []]
        rules.append({
            "id": p.get("id", "unknown"),
            "when_all": when_all,
            "pattern": re.compile(p["pattern"]),
            "replacement": p.get("replacement", ""),
            "count": int(p.get("count_per_line", 1)),
        })
    return rules

## Step 15 — look at what got compiled

Compile the example YAML above and print the resulting rule list — real
output before this notebook applies any of these rules to text.

In [ ]:
rules = _compile_pitfalls(_EXAMPLE_PITFALLS_YAML)
print(f"{len(rules)} rule(s) compiled:")
for r in rules:
    print(f"  id={r['id']!r} pattern={r['pattern'].pattern!r} replacement={r['replacement']!r}")

## Step 16 — `apply_ocr_pitfalls`, wiring the compiled rules into text

Applies each compiled rule to text, line by line, skipping a rule on a
line that doesn't match its `when_line_matches_all` guard.

In [ ]:
def apply_ocr_pitfalls(text: str, rules: list[dict]) -> str:
    out_lines = []
    for line in text.split("\n"):
        s = line
        for rule in rules:
            if rule["when_all"] and not all(w.search(s) for w in rule["when_all"]):
                continue
            s = rule["pattern"].sub(rule["replacement"], s, count=rule["count"])
        out_lines.append(s)
    return "\n".join(out_lines)

## Step 17 — run it on a real example and look at the fix

`example_vision_output` stands in for what a working vision call might
have returned — a bare `*` where the prompt asked for `★`.

In [ ]:
example_vision_output = "Final answer: x = 5 *"  # what a working call might have returned
print("before:", repr(example_vision_output))
print("after: ", repr(apply_ocr_pitfalls(example_vision_output, rules)))

## What this stage covers, what it doesn't

| Kept | Left out | Why |
|---|---|---|
| Both prompts, the two-pass transcribe→SVG shape, JPEG re-encoding | a config-settings abstraction layer | model ids read from `os.environ` directly instead |
| the YAML-rule compile-and-apply mechanism | the full pitfalls registry | tuned to a real, dataset-specific set of student answer scripts |
| the exact retired model id, named and dated | nothing — the failure is the point | see the first cell |

**The open task:** pick a vision-capable model that's live today, point
`GROQ_VISION_MODEL` (or a rewritten `_call_groq_vision`) at it, and confirm
it transcribes `sample-data/handwriting-sample.png`. See `03-orientation
.ipynb` for the other image-only path this stage covers, and the stage
`README.md` for the full run report.